### What does `source setup.sh` mean?

`source setup.sh` runs the script in your **current shell session**.

- Environment changes (like `conda activate`, exported variables, and PATH updates) persist in the current terminal.
- This differs from `./setup.sh`, which usually runs in a subshell, so many environment changes are not kept after it exits.
- In this project, it is typically used to create/configure the `cs224n_dfp` environment and install dependencies.

Equivalent command:

```bash
. setup.sh
```


# 5.1 Details of the GPT-2 Model

GPT-2 是一个 **decoder-only** Transformer 模型，使用前序 token 序列预测下一个 token。

---

## Tokenization

GPT-2 使用 **Byte Pair Encoding (BPE)** 分词。BPE 将单词分解为子词单元（subword tokens），例如：

```
"Hello, my name is Chris"
→ ["Hel", "lo", ",", "my", "name", "is", "C", "hris"]
```

---

## Embedding Layer（嵌入层）

**Figure 1：GPT-2 嵌入层结构**

```
Input Sentence:  "Hello, my name is Chris"
        ↓  BPE Tokenize
Tokens:  [Hel] [lo] [,] [my] [name] [is] [C] [hris]
        ↓
Token Embeddings:    v₁   v₂   v₃   v₄   v₅   v₆   v₇   v₈   ∈ ℝᴰ
        +
Position Embeddings: p₁   p₂   p₃   p₄   p₅   p₆   p₇   p₈   ∈ ℝᴰ
        ↓
Output Embeddings:   e₁   e₂   e₃   e₄   e₅   e₆   e₇   e₈
```

**说明：**
- 输入 token indices $w_1, \ldots, w_k \in \mathbb{N}$，通过 embedding lookup 映射为 token 嵌入向量 $v_1, \ldots, v_k \in \mathbb{R}^D$
- 位置嵌入（positional embeddings）是**可学习**的，为每个位置 $1, \ldots, 1024$ 分别学习一个向量
- 最终输入嵌入 = **token embedding + position embedding**
- 嵌入维度 $D = 768$，最大上下文长度（context length）= **1024**

---

## GPT-2 Transformer Layer（GPT-2 Transformer 层）

**Figure 2：单个 GPT-2 Transformer 层结构**

```
        ┌──────────────────────────────────┐
        │                                  │
 x ─────┤─────────────────────────────┐    │
        │  LayerNorm(x)               │    │
        │       ↓                     │    │
        │  Masked Multi-Head          │    │
        │  Self-Attention             │    │
        │       ↓                     │    │
        │  Dropout                    │    │
        │       ↓                     │    │
        │    + ←──────────────────────┘    │  ← Residual connection
        │       ↓                     ┐    │
        │  LayerNorm                  │    │
        │       ↓                     │    │
        │  MLP (Feed-Forward)         │    │
        │       ↓                     │    │
        │  Dropout                    │    │
        │       ↓                     │    │
        │    + ←──────────────────────┘    │  ← Residual connection
        │       ↓                          │
        └──────────────── Output ──────────┘
```

GPT-2 (small) 将上述结构**堆叠 12 层**（`config.num_hidden_layers = 12`）。

---

## Multi-Head Self-Attention（多头自注意力）

**Figure 3：缩放点积注意力 & 多头注意力结构**

```
┌─────────────────────────────────┐   ┌──────────────────────────────────────┐
│  Scaled Dot-Product Attention   │   │       Multi-Head Attention           │
│                                 │   │                                      │
│   Q ──┐                         │   │  Q ──► Linear  K ──► Linear          │
│   K ──┼──► MatMul(Q,Kᵀ)         │   │  V ──► Linear                        │
│   V ──┘       ↓                 │   │        ↓        ↓        ↓           │
│           Scale (÷√dₖ)          │   │      head₁   head₂  ... headₕ        │
│               ↓                 │   │        ↓        ↓        ↓           │
│           (Mask)                │   │        └────────┴────────┘           │
│               ↓                 │   │               Concat                  │
│           Softmax               │   │                 ↓                    │
│               ↓                 │   │           Linear (W_O)               │
│           MatMul ──► Output     │   │                 ↓                    │
│               ↑                 │   │              Output                  │
│               V                 │   │                                      │
└─────────────────────────────────┘   └──────────────────────────────────────┘
```

**缩放点积注意力公式：**

$$\text{Attention}(Q, K, V) = \text{Softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**多头注意力公式：**

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\, W^O$$

$$\text{where} \quad \text{head}_i = \text{Attention}(QW_i^Q,\; KW_i^K,\; VW_i^V)$$

**投影矩阵维度：**

| 矩阵 | 维度 |
|------|------|
| $W_i^Q$ | $\mathbb{R}^{d_{\text{model}} \times d_k}$ |
| $W_i^K$ | $\mathbb{R}^{d_{\text{model}} \times d_k}$ |
| $W_i^V$ | $\mathbb{R}^{d_{\text{model}} \times d_v}$ |
| $W^O$   | $\mathbb{R}^{h d_v \times d_{\text{model}}}$ |

---

## Masked Multi-Head Self-Attention（因果掩码多头注意力）

GPT-2 使用**因果掩码（causal mask）** 防止 token 关注到未来位置，避免模型在训练时"偷看"答案（即下一个 token）。

**Figure 4：因果掩码示意（softmax 之前的注意力权重矩阵）**

```
            Hel  lo   ,   my  name  is   C  hris   .
Hel      [   ✓    ✗    ✗    ✗    ✗    ✗    ✗    ✗    ✗  ]
lo       [   ✓    ✓    ✗    ✗    ✗    ✗    ✗    ✗    ✗  ]
,        [   ✓    ✓    ✓    ✗    ✗    ✗    ✗    ✗    ✗  ]
my       [   ✓    ✓    ✓    ✓    ✗    ✗    ✗    ✗    ✗  ]
name     [   ✓    ✓    ✓    ✓    ✓    ✗    ✗    ✗    ✗  ]
is       [   ✓    ✓    ✓    ✓    ✓    ✓    ✗    ✗    ✗  ]
C        [   ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✗    ✗  ]
hris     [   ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✗  ]
.        [   ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ]

✓ = 可关注（保留注意力权重）
✗ = 被遮盖（softmax 前设为 -∞，softmax 后权重 ≈ 0）
```

**实现方式：** 使用 `torch.triu` 生成上三角掩码，在 softmax **之前**加到注意力分数矩阵上：

$$\text{score}_{ij} = \frac{Q_i K_j^T}{\sqrt{d_k}} + M_{ij}, \qquad M_{ij} = \begin{cases} 0 & i \geq j \\ -\infty & i < j \end{cases}$$

---

## Position-wise Feed-Forward Network（逐位置前馈网络）

每个 Transformer 层还包含一个两层 MLP，中间用 **ReLU** 激活：

$$\text{FFN}(x) = \max(0,\; xW_1 + b_1)\, W_2 + b_2$$

---

## Dropout

GPT-2 在以下位置应用 Dropout，丢弃率 $p_{\text{drop}} = 0.1$：

1. **注意力层**输出之后（残差连接之前）
2. **MLP 层**输出之后（残差连接之前）
3. **嵌入层**输出（token emb + position emb 求和）之后

---

## GPT-2 Output（模型输出）

GPT-2 整体结构：

```
Input tokens
     ↓
[Embedding Layer]  token_embedding + pos_embedding  → dropout
     ↓
[GPT-2 Layer 1]    (LayerNorm → Masked MHA → Residual → LayerNorm → FFN → Residual)
     ↓
[GPT-2 Layer 2]
     ↓
    ...
     ↓
[GPT-2 Layer 12]
     ↓
last_hidden_state  (shape: batch × seq_len × d_model)
last_token         (shape: batch × d_model)  ← 用于分类任务
```

---

## Training GPT-2（训练目标）

GPT-2 以**自回归 next-token prediction** 为训练目标，最大化序列的对数似然：

$$\log P(x_1, x_2, \ldots, x_n) = \log \prod_{i=1}^{n} P(x_i \mid x_1, x_2, \ldots, x_{i-1}) = \sum_{i=1}^{n} \log P(x_i \mid x_1, \ldots, x_{i-1})$$

**关键思路：**
- 在海量无标注文本上通过 next-token prediction 预训练，学习通用语言理解能力
- 之后通过 **fine-tuning** 迁移到情感分类、同义句检测、诗歌生成等下游任务
- GPT-2 (small)：12 层，768 维，$d_k = d_v = 64$，12 个注意力头，共约 **117M 参数**

## CausalSelfAttention 代码笔记（`modules/attention.py`）

目标：实现 GPT-2 的 **Masked Multi-Head Self-Attention**：

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + \text{mask}\right)V$$

### 张量形状（多头）

- 输入 `hidden_states`: `[bs, t, d_model]`
- 线性投影后 `Q/K/V`: `[bs, h, t, d_k]`，其中 `d_k = d_model / h`
- 注意力分数 `scores = Q @ K^T`: `[bs, h, t, t]`

### 为什么要 `key.transpose(-2, -1)`

`K` 的形状是 `[bs, h, t, d_k]`，转置最后两维后变成 `[bs, h, d_k, t]`，才能让：

- `Q @ K^T`: `[bs,h,t,d_k] @ [bs,h,d_k,t] -> [bs,h,t,t]`

### Causal mask（因果掩码：不能看未来）

- 规则：位置 `i` 只能关注 `j <= i`，遮住 `j > i`。
- 构造：`torch.triu(ones(t,t), diagonal=1)` 得到严格上三角（未来位置）为 True。
- 应用：`scores = scores.masked_fill(causal_mask, -inf)`（必须在 softmax 之前）。

### Padding mask（padding 掩码：不能看 pad token）

- `attention_mask`（来自 `get_extended_attention_mask`）是 **additive mask**：非 pad 为 0，pad 为大负数（如 -10000）。
- 形状是 `[bs, 1, 1, t]`，会广播到 `[bs, h, t, t]`：`scores = scores + attention_mask`。

### softmax 的 `dim=-1`

- `attn_weights = softmax(scores, dim=-1)` 表示对最后一维（key 维 `j`）做归一化。
- 直观上：每个 query 位置 `i` 的一整行权重和为 1。

### 输出

- `context = attn_weights @ V` 得到 `[bs, h, t, d_k]`。
- 合并 heads：`rearrange('b h t d -> b t (h d)')` 得到 `[bs, t, d_model]`。
- GPT-2 的输出投影 `W_O`（notes 里 MultiHead 最后的 Linear）在本工程由 `GPT2Layer.attention_dense` 负责，而不在 `CausalSelfAttention` 内部做。


**Optimizer test passed!** 通过了。来看完整的实现逻辑：

```29:67:optimizer.py
    def step(self, closure: Callable = None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad.data
                ...
                # Initialize state on first step
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p.data)     # m_0: 1st moment
                    state['exp_avg_sq'] = torch.zeros_like(p.data)  # v_0: 2nd moment
                ...
```

---

## 对应 PDF 算法的每一步

| 代码 | 对应公式 | 说明 |
|------|---------|------|
| `m.mul_(beta1).add_(grad, alpha=1-beta1)` | $m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t$ | 一阶矩（梯度均值）滑动平均 |
| `v.mul_(beta2).addcmul_(grad, grad, value=1-beta2)` | $v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$ | 二阶矩（梯度方差）滑动平均 |
| `alpha_t = alpha * sqrt(1-β₂ᵗ) / (1-β₁ᵗ)` | $\alpha_t = \alpha \cdot \frac{\sqrt{1-\beta_2^t}}{1-\beta_1^t}$ | **高效版** bias correction，直接折进学习率 |
| `p.data.addcdiv_(m, v.sqrt().add_(eps), value=-alpha_t)` | $\theta_t = \theta_{t-1} - \alpha_t \cdot \frac{m_t}{\sqrt{v_t}+\epsilon}$ | 参数更新 |
| `p.data.add_(p.data, alpha=-alpha * weight_decay)` | $\theta_t = \theta_t - \alpha \lambda \theta_t$ | **解耦**权重衰减（用原始 $\alpha$，不用 $\alpha_t$） |

**AdamW 和 Adam 的关键区别** 就在最后一步：权重衰减直接作用于参数本身（$-\alpha\lambda\theta$），而不是加到梯度里，因此不受 bias correction 影响，这就是"Decoupled Weight Decay"的含义。